<div align="center">

<h2>ESCUELA POLITÉCNICA NACIONAL</h2>

<h3>FACULTAD DE INGENIERÍA DE SISTEMAS</h3>

<p><strong>Materia:</strong> Recuperación de la Información</p>

<p><strong>Estudiante:</strong> Estéfano Condoy</p>

</div>

<hr>

# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [7]:
# Aquí importo las herramientas para preparar el corpus y mostrar las tablas.
import re
from collections import Counter
from math import log, log1p

import nltk
import numpy as np
import pandas as pd
from IPython.display import display
from nltk.corpus import gutenberg

# Aquí cargo los 18 documentos de Gutenberg que usé en el deber anterior.
try:
    ids_documentos = list(map(str, gutenberg.fileids()))
except LookupError:
    nltk.download("gutenberg", raise_on_error=True)
    ids_documentos = list(map(str, gutenberg.fileids()))

# Aquí aplico el mismo preprocesamiento: minúsculas y apóstrofos internos.
TOKEN_RE = re.compile(r"[^\W_]+(?:'[^\W_]+)?", flags=re.UNICODE)

def tokenizar(texto):
    return TOKEN_RE.findall(texto.lower().replace("\u2019", "'"))

# Aquí guardo el ID, el título del encabezado y el texto original de cada libro.
filas_corpus = []
for doc_id in ids_documentos:
    texto = gutenberg.raw(doc_id)
    encabezado = texto.lstrip().splitlines()[0].strip().strip("[]")
    titulo = encabezado.split(" by ", 1)[0].rstrip(",")
    filas_corpus.append({"doc_id": doc_id, "Título": titulo, "Texto": texto})

corpus = pd.DataFrame(filas_corpus)
corpus["tokens"] = corpus["Texto"].map(tokenizar)
titulos = dict(zip(corpus["doc_id"], corpus["Título"]))
indice_documento = {doc_id: i for i, doc_id in enumerate(ids_documentos)}
total_documentos = len(corpus)

# Aquí calculo las longitudes en tokens y la longitud promedio de la colección.
longitudes = corpus["tokens"].map(len).to_numpy()
longitud_media = longitudes.mean()

# Aquí cuento TF por documento y sumo uno al DF por cada documento con el término.
tf_documentos = [Counter(tokens) for tokens in corpus["tokens"]]
df = Counter()
for conteos in tf_documentos:
    df.update(conteos.keys())

# Aquí calculo los dos IDF una sola vez con el DF de todo el corpus.
idf_tfidf = {termino: log(total_documentos / frecuencia) + 1
             for termino, frecuencia in df.items()}
# Aquí uso para BM25 el IDF con log(1 + x) documentado por Apache Lucene.
# https://lucene.apache.org/core/10_5_2/core/org/apache/lucene/search/similarities/BM25Similarity.html
idf_bm25 = {termino: log1p((total_documentos - frecuencia + 0.5) / (frecuencia + 0.5))
            for termino, frecuencia in df.items()}

# Aquí muestro la tabla y un ejemplo de los DF e IDF calculados.
with pd.option_context("display.max_colwidth", 55):
    display(corpus)
print(f"Documentos: {total_documentos} | Vocabulario: {len(df):,} términos")
print(f"Longitud media del corpus: {longitud_media:,.2f} tokens")
display(pd.DataFrame([
    {"Término": t, "DF": df[t], "IDF TF-IDF": idf_tfidf[t], "IDF BM25": idf_bm25[t]}
    for t in ("time", "whale", "queequeg")
]).round(6))

,doc_id,Título,Texto,tokens
0,austen-emma.txt,Emma,[Emma by Jane Austen 1816]\n\nVOLUME I\n\nCHAPTER I...,"[emma, by, jane, austen, 1816, volume, i, chapter, ..."
1,austen-persuasion.txt,Persuasion,[Persuasion by Jane Austen 1818]\n\n\nChapter 1\n\n...,"[persuasion, by, jane, austen, 1818, chapter, 1, si..."
2,austen-sense.txt,Sense and Sensibility,[Sense and Sensibility by Jane Austen 1811]\n\nCHAP...,"[sense, and, sensibility, by, jane, austen, 1811, c..."
3,bible-kjv.txt,The King James Bible,[The King James Bible]\n\nThe Old Testament of the ...,"[the, king, james, bible, the, old, testament, of, ..."
4,blake-poems.txt,Poems,[Poems by William Blake 1789]\n\n \nSONGS OF INNOCE...,"[poems, by, william, blake, 1789, songs, of, innoce..."
5,bryant-stories.txt,Stories to Tell to Children,[Stories to Tell to Children by Sara Cone Bryant 19...,"[stories, to, tell, to, children, by, sara, cone, b..."
6,burgess-busterbrown.txt,The Adventures of Buster Bear,[The Adventures of Buster Bear by Thornton W. Burge...,"[the, adventures, of, buster, bear, by, thornton, w..."
7,carroll-alice.txt,Alice's Adventures in Wonderland,[Alice's Adventures in Wonderland by Lewis Carroll ...,"[alice's, adventures, in, wonderland, by, lewis, ca..."
8,chesterton-ball.txt,The Ball and The Cross,[The Ball and The Cross by G.K. Chesterton 1909]\n\...,"[the, ball, and, the, cross, by, g, k, chesterton, ..."
9,chesterton-brown.txt,The Wisdom of Father Brown,[The Wisdom of Father Brown by G. K. Chesterton 191...,"[the, wisdom, of, father, brown, by, g, k, chestert..."


Documentos: 18 | Vocabulario: 43,825 términos
Longitud media del corpus: 121,225.78 tokens


,Término,DF,IDF TF-IDF,IDF BM25
0,time,18,1.000000,0.026668
1,whale,6,2.098612,1.072637
2,queequeg,1,3.890372,2.538974


**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Aquí escojo ocho consultas en inglés y escribo qué busca el usuario.
consultas = {
    "time": ("Pasajes que hablen del tiempo.",
             "Un término presente en los 18 documentos."),
    "whale ship captain": ("Relatos sobre ballenas y capitanes de barco.",
                            "Tres términos con distinto alcance. whale tiene el menor DF."),
    "marriage family fortune": ("Novelas sobre matrimonio y patrimonio familiar.",
                                "Tres términos con DF similares, entre 12 y 13."),
    "king crown battle": ("Historias de reyes, coronas y batallas.",
                          "king y crown son más frecuentes en el corpus que battle."),
    "faith prayer god heaven angels": ("Pasajes sobre fe, oración y el cielo.",
                                       "Cinco términos con DF entre 10 y 16."),
    "queequeg": ("Pasajes donde aparece el personaje Queequeg.",
                 "Un nombre propio que solo aparece en un documento."),
    "cheshire gryphon": ("Pasajes sobre el gato de Cheshire o el grifo de Alicia.",
                          "Dos términos raros con DF de 3 y 2."),
    "thermonuclear microprocessor": ("Textos sobre tecnología termonuclear o microprocesadores.",
                                     "Caso de control con DF cero para ambos términos."),
}
tokens_consultas = {consulta: tokenizar(consulta) for consulta in consultas}
display(pd.DataFrame([
    {
        "Consulta": consulta,
        "Qué busca el usuario": intencion,
        "Longitud": len(tokens_consultas[consulta]),
        "DF por término": ", ".join(f"{t}={df[t]}" for t in tokens_consultas[consulta]),
        "Justificación por DF": motivo,
    }
    for consulta, (intencion, motivo) in consultas.items()
]))

# Aquí ajusto TF-IDF con el mismo tokenizador y conservo la matriz dispersa.
vectorizador = TfidfVectorizer(
    analyzer=tokenizar, use_idf=True, smooth_idf=False, sublinear_tf=False, norm=None
)
tfidf_documentos = vectorizador.fit_transform(corpus["Texto"]).tocsr()
vocabulario = vectorizador.get_feature_names_out()
np.testing.assert_allclose(vectorizador.idf_, [idf_tfidf[t] for t in vocabulario])

# Aquí transformo las consultas con el vocabulario y los pesos del corpus.
tfidf_consultas = vectorizador.transform(list(consultas))
cosenos = cosine_similarity(
    tfidf_consultas, tfidf_documentos, dense_output=False
).tocsr()

# Aquí calculo el aporte de cada término distinto y sumo los aportes por documento.
K1_BASE, B_BASE = 1.2, 0.75

def calcular_bm25(tokens, k1=K1_BASE, b=B_BASE):
    # Aquí b pondera la longitud relativa y k1 controla la saturación de TF.
    ajuste_longitud = k1 * (1 - b + b * longitudes / longitud_media)
    aportes = {}
    for termino in dict.fromkeys(tokens):
        if termino not in idf_bm25:
            continue
        tf = np.array([conteos[termino] for conteos in tf_documentos])
        aportes[termino] = idf_bm25[termino] * (k1 + 1) * tf / (tf + ajuste_longitud)
    puntuaciones = sum(aportes.values(), np.zeros(total_documentos))
    return puntuaciones, aportes

# Aquí descarto scores cero y desempato por el ID de texto en orden ascendente.
def ordenar_positivos(pares):
    return sorted(
        [(doc_id, float(score)) for doc_id, score in pares if score > 0],
        key=lambda par: (-par[1], par[0]),
    )

# Aquí extraigo únicamente los cosenos almacenados y preparo ambos top 10.
rankings_tfidf, rankings_bm25, aportes_base = {}, {}, {}
for i, (consulta, tokens) in enumerate(tokens_consultas.items()):
    fila_coseno = cosenos[i:i + 1]
    pares_coseno = zip(
        (ids_documentos[j] for j in fila_coseno.indices), fila_coseno.data
    )
    scores_bm25, aportes_base[consulta] = calcular_bm25(tokens)
    rankings_tfidf[consulta] = ordenar_positivos(pares_coseno)[:10]
    rankings_bm25[consulta] = ordenar_positivos(zip(ids_documentos, scores_bm25))[:10]

,Consulta,Qué busca el usuario,Longitud,DF por término,Justificación por DF
0,time,Pasajes que hablen del tiempo.,1,time=18,Un término presente en los 18 documentos.
1,whale ship captain,Relatos sobre ballenas y capitanes de barco.,3,"whale=6, ship=13, captain=9",Tres términos con distinto alcance. whale tien...
2,marriage family fortune,Novelas sobre matrimonio y patrimonio familiar.,3,"marriage=12, family=13, fortune=12","Tres términos con DF similares, entre 12 y 13."
3,king crown battle,"Historias de reyes, coronas y batallas.",3,"king=15, crown=15, battle=9",king y crown son más frecuentes en el corpus q...
4,faith prayer god heaven angels,"Pasajes sobre fe, oración y el cielo.",5,"faith=15, prayer=10, god=16, heaven=13, angels=11",Cinco términos con DF entre 10 y 16.
5,queequeg,Pasajes donde aparece el personaje Queequeg.,1,queequeg=1,Un nombre propio que solo aparece en un docume...
6,cheshire gryphon,Pasajes sobre el gato de Cheshire o el grifo d...,2,"cheshire=3, gryphon=2",Dos términos raros con DF de 3 y 2.
7,thermonuclear microprocessor,Textos sobre tecnología termonuclear o micropr...,2,"thermonuclear=0, microprocessor=0",Caso de control con DF cero para ambos términos.


**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [16]:
from itertools import zip_longest

# Aquí coloco los dos rankings en una tabla y comparo sus IDs.
def etiqueta_documento(doc_id):
    return f"{doc_id} — {titulos[doc_id]}" if doc_id else "—"

columnas_ranking = ["Posición", "ID y título TF-IDF", "Coseno",
                    "ID y título BM25", "Score BM25"]
resumen = []
for consulta in consultas:
    top_tfidf = rankings_tfidf[consulta]
    top_bm25 = rankings_bm25[consulta]
    filas = []
    for posicion, (resultado_tfidf, resultado_bm25) in enumerate(
        zip_longest(top_tfidf, top_bm25, fillvalue=(None, None)), start=1
    ):
        id_tfidf, coseno = resultado_tfidf
        id_bm25, score_bm25 = resultado_bm25
        filas.append([posicion, etiqueta_documento(id_tfidf), coseno,
                      etiqueta_documento(id_bm25), score_bm25])

    print(f"Consulta: {consulta!r}")
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(filas, columns=columnas_ranking).round(6))

    ids_tfidf = {doc_id for doc_id, _ in top_tfidf}
    ids_bm25 = {doc_id for doc_id, _ in top_bm25}
    resumen.append({
        "Consulta": consulta,
        "Tamaño TF-IDF": len(top_tfidf),
        "Tamaño BM25": len(top_bm25),
        "IDs en común": len(ids_tfidf & ids_bm25),
        "Primer ID TF-IDF": top_tfidf[0][0] if top_tfidf else "—",
        "Primer ID BM25": top_bm25[0][0] if top_bm25 else "—",
    })

print("Resumen de tamaños, coincidencias y primeros resultados")
display(pd.DataFrame(resumen))

# Aquí elijo dos documentos cuyo orden relativo cambia entre los dos métodos.
consulta_detalle = "whale ship captain"
id_persuasion, id_bryant = "austen-persuasion.txt", "bryant-stories.txt"
ids_detalle = (id_persuasion, id_bryant)
tokens_detalle = tokens_consultas[consulta_detalle]
aportes_detalle = aportes_base[consulta_detalle]

detalle = []
for doc_id in ids_detalle:
    i = indice_documento[doc_id]
    for termino in tokens_detalle:
        detalle.append({
            "doc_id": doc_id,
            "Longitud (tokens)": longitudes[i],
            "Término": termino,
            "TF en documento": tf_documentos[i][termino],
            "DF en corpus": df[termino],
            "IDF BM25": idf_bm25[termino],
            "Aporte BM25": aportes_detalle[termino][i],
        })

print(f"Desglose de BM25 para {consulta_detalle!r}")
display(pd.DataFrame(detalle).round(6))

# Aquí uso los scores de cada método para explicar su propio orden.
tfidf = dict(rankings_tfidf[consulta_detalle])
bm25 = dict(rankings_bm25[consulta_detalle])
print(f"TF-IDF ubica a Persuasion en el puesto 2 ({tfidf[id_persuasion]:.6f}) "
      f"y a Stories to Tell to Children en el 3 ({tfidf[id_bryant]:.6f}). "
      "El peso de captain, que aparece 303 veces en Persuasion frente a 2 "
      "en el otro libro, alinea más su vector con la consulta.")
print(f"BM25 ubica a Stories to Tell to Children en el puesto 2 ({bm25[id_bryant]:.6f}) "
      f"y a Persuasion en el 8 ({bm25[id_persuasion]:.6f}). "
      "Las 303 repeticiones de captain tienen un efecto limitado por la saturación. "
      "Stories to Tell to Children es más corto y contiene los tres términos. "
      "Solo whale, que aparece en 6 documentos, le aporta 2.2685 puntos, "
      "más que el score total de Persuasion.")

Consulta: 'time'


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,shakespeare-macbeth.txt — The Tragedie of Macbeth,0.027807,edgeworth-parents.txt — The Parent's Assistant,0.058385
1,2,burgess-busterbrown.txt — The Adventures of Buster Bear,0.026402,austen-sense.txt — Sense and Sensibility,0.058379
2,3,carroll-alice.txt — Alice's Adventures in Wonderland,0.024298,austen-emma.txt — Emma,0.058358
3,4,austen-sense.txt — Sense and Sensibility,0.020999,melville-moby_dick.txt — Moby Dick,0.058337
4,5,edgeworth-parents.txt — The Parent's Assistant,0.020684,austen-persuasion.txt — Persuasion,0.058317
5,6,austen-persuasion.txt — Persuasion,0.018999,bryant-stories.txt — Stories to Tell to Children,0.058283
6,7,bryant-stories.txt — Stories to Tell to Children,0.018784,carroll-alice.txt — Alice's Adventures in Wonderland,0.058261
7,8,austen-emma.txt — Emma,0.018476,whitman-leaves.txt — Leaves of Grass,0.058246
8,9,shakespeare-hamlet.txt — The Tragedie of Hamlet,0.016140,burgess-busterbrown.txt — The Adventures of Buster Bear,0.058163
9,10,melville-moby_dick.txt — Moby Dick,0.015652,shakespeare-macbeth.txt — The Tragedie of Macbeth,0.058124


Consulta: 'whale ship captain'


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,melville-moby_dick.txt — Moby Dick,0.101403,melville-moby_dick.txt — Moby Dick,4.619866
1,2,austen-persuasion.txt — Persuasion,0.036574,bryant-stories.txt — Stories to Tell to Children,4.088124
2,3,bryant-stories.txt — Stories to Tell to Children,0.005477,whitman-leaves.txt — Leaves of Grass,4.068482
3,4,whitman-leaves.txt — Leaves of Grass,0.005000,bible-kjv.txt — The King James Bible,2.690243
4,5,chesterton-brown.txt — The Wisdom of Father Brown,0.003710,shakespeare-hamlet.txt — The Tragedie of Hamlet,2.469083
5,6,chesterton-ball.txt — The Ball and The Cross,0.002969,chesterton-ball.txt — The Ball and The Cross,2.355762
6,7,edgeworth-parents.txt — The Parent's Assistant,0.002488,chesterton-brown.txt — The Wisdom of Father Brown,2.173682
7,8,bible-kjv.txt — The King James Bible,0.001678,austen-persuasion.txt — Persuasion,2.172090
8,9,shakespeare-hamlet.txt — The Tragedie of Hamlet,0.001504,edgeworth-parents.txt — The Parent's Assistant,1.961575
9,10,chesterton-thursday.txt — The Man Who Was Thursday,0.000407,chesterton-thursday.txt — The Man Who Was Thursday,1.549889


Consulta: 'marriage family fortune'


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,austen-persuasion.txt — Persuasion,0.013141,austen-sense.txt — Sense and Sensibility,2.531286
1,2,austen-sense.txt — Sense and Sensibility,0.011320,austen-persuasion.txt — Persuasion,2.526870
2,3,austen-emma.txt — Emma,0.007774,austen-emma.txt — Emma,2.508096
3,4,shakespeare-hamlet.txt — The Tragedie of Hamlet,0.004853,chesterton-thursday.txt — The Man Who Was Thursday,2.174590
4,5,chesterton-brown.txt — The Wisdom of Father Brown,0.004641,whitman-leaves.txt — Leaves of Grass,2.143200
5,6,shakespeare-macbeth.txt — The Tragedie of Macbeth,0.003499,bryant-stories.txt — Stories to Tell to Children,2.108529
6,7,edgeworth-parents.txt — The Parent's Assistant,0.003393,shakespeare-hamlet.txt — The Tragedie of Hamlet,1.723221
7,8,bryant-stories.txt — Stories to Tell to Children,0.002598,edgeworth-parents.txt — The Parent's Assistant,1.596178
8,9,shakespeare-caesar.txt — The Tragedie of Julius Caesar,0.001984,shakespeare-caesar.txt — The Tragedie of Julius Caesar,1.461789
9,10,chesterton-thursday.txt — The Man Who Was Thursday,0.001547,melville-moby_dick.txt — Moby Dick,1.429701


... (11 outputs left out)

,Consulta,Tamaño TF-IDF,Tamaño BM25,IDs en común,Primer ID TF-IDF,Primer ID BM25
0,time,10,10,9,shakespeare-macbeth.txt,edgeworth-parents.txt
1,whale ship captain,10,10,10,melville-moby_dick.txt,melville-moby_dick.txt
2,marriage family fortune,10,10,8,austen-persuasion.txt,austen-sense.txt
3,king crown battle,10,10,7,shakespeare-hamlet.txt,bible-kjv.txt
4,faith prayer god heaven angels,10,10,9,milton-paradise.txt,bible-kjv.txt
5,queequeg,1,1,1,melville-moby_dick.txt,melville-moby_dick.txt
6,cheshire gryphon,4,4,4,carroll-alice.txt,carroll-alice.txt
7,thermonuclear microprocessor,0,0,0,—,—


Desglose de BM25 para 'whale ship captain'


,doc_id,Longitud (tokens),Término,TF en documento,DF en corpus,IDF BM25,Aporte BM25
0,austen-persuasion.txt,83660,whale,0,6,1.072637,0.000000
1,austen-persuasion.txt,83660,ship,6,13,0.341749,0.651788
2,austen-persuasion.txt,83660,captain,303,9,0.693147,1.520302
3,bryant-stories.txt,46321,whale,16,6,1.072637,2.268508
4,bryant-stories.txt,46321,ship,5,13,0.341749,0.666072
5,bryant-stories.txt,46321,captain,2,9,0.693147,1.153544


TF-IDF ubica a Persuasion en el puesto 2 (0.036574) y a Stories to Tell to Children en el 3 (0.005477). El peso de captain, que aparece 303 veces en Persuasion frente a 2 en el otro libro, alinea más su vector con la consulta.
BM25 ubica a Stories to Tell to Children en el puesto 2 (4.088124) y a Persuasion en el 8 (2.172090). Las 303 repeticiones de captain tienen un efecto limitado por la saturación. Stories to Tell to Children es más corto y contiene los tres términos. Solo whale, que aparece en 6 documentos, le aporta 2.2685 puntos, más que el score total de Persuasion.


**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [23]:
# Aquí mantengo la misma consulta y cambio un parámetro por vez.
configuraciones = {
    "Base": (K1_BASE, B_BASE),
    "Sin ajuste por longitud": (K1_BASE, 0.0),
    "Mayor efecto de repeticiones": (2.0, B_BASE),
}
id_whitman = "whitman-leaves.txt"
ids_experimento = (id_bryant, id_whitman)
posiciones_parametros = []
aportes_parametros = {}
scores_parametros = {}

# Aquí comparo los mismos dos libros en los rankings completos de BM25.
for nombre, (k1, b) in configuraciones.items():
    scores, aportes = calcular_bm25(tokens_detalle, k1=k1, b=b)
    ranking = ordenar_positivos(zip(ids_documentos, scores))
    posiciones = {doc_id: puesto for puesto, (doc_id, _) in enumerate(ranking, start=1)}
    scores_parametros[nombre] = dict(ranking)
    aportes_parametros[nombre] = aportes
    for doc_id in ids_experimento:
        i = indice_documento[doc_id]
        posiciones_parametros.append({
            "Ejecución": nombre,
            "k1": k1,
            "b": b,
            "doc_id": doc_id,
            "Longitud (tokens)": longitudes[i],
            "Posición": posiciones[doc_id],
            "Score BM25": scores[i],
        })

print(f"Consulta: {consulta_detalle!r} | Longitud media: {longitud_media:,.2f} tokens")
display(pd.DataFrame(posiciones_parametros).round(6))

# Aquí desgloso los aportes para ver qué cambia aunque se conserve la posición.
filas_aportes = []
for doc_id in ids_experimento:
    i = indice_documento[doc_id]
    for termino in tokens_detalle:
        fila = {"doc_id": doc_id, "Término": termino, "TF": tf_documentos[i][termino]}
        for nombre, aportes in aportes_parametros.items():
            fila[nombre] = aportes[termino][i]
        filas_aportes.append(fila)
display(pd.DataFrame(filas_aportes).round(6))

# Aquí interpreto el cambio de b con las longitudes y scores medidos.
i_bryant = indice_documento[id_bryant]
i_whitman = indice_documento[id_whitman]
print(f"Con b=0, el libro de Bryant ({longitudes[i_bryant]:,} tokens) pierde "
      "la ventaja por ser más corto que la media. Leaves of Grass "
      f"({longitudes[i_whitman]:,} tokens) deja de recibir la penalización por longitud. "
      "Por eso intercambian los puestos 2 y 3.")

# Aquí comparo el efecto de k1 sobre una frecuencia alta y una baja del mismo libro.
base = aportes_parametros["Base"]
mayor_k1 = aportes_parametros["Mayor efecto de repeticiones"]
razon_base = base["whale"][i_bryant] / base["captain"][i_bryant]
razon_mayor = mayor_k1["whale"][i_bryant] / mayor_k1["captain"][i_bryant]
scores_mayor = scores_parametros["Mayor efecto de repeticiones"]
print("Con k1=2, la saturación es más lenta. En el libro de Bryant, whale (TF=16) "
      f"pasa de aportar {razon_base:.2f} a {razon_mayor:.2f} veces lo que aporta "
      "captain (TF=2). El cambio no invierte el orden de estos dos libros: "
      f"Bryant sigue por encima con {scores_mayor[id_bryant]:.6f} frente a "
      f"{scores_mayor[id_whitman]:.6f} de Whitman.")

Consulta: 'whale ship captain' | Longitud media: 121,225.78 tokens


,Ejecución,k1,b,doc_id,Longitud (tokens),Posición,Score BM25
0,Base,1.2,0.75,bryant-stories.txt,46321,2,4.088124
1,Base,1.2,0.75,whitman-leaves.txt,124337,3,4.068482
2,Sin ajuste por longitud,1.2,0.00,bryant-stories.txt,46321,3,3.754570
3,Sin ajuste por longitud,1.2,0.00,whitman-leaves.txt,124337,2,4.077408
4,Mayor efecto de repeticiones,2.0,0.75,bryant-stories.txt,46321,2,5.213018
5,Mayor efecto de repeticiones,2.0,0.75,whitman-leaves.txt,124337,3,5.162012


,doc_id,Término,TF,Base,Sin ajuste por longitud,Mayor efecto de repeticiones
0,bryant-stories.txt,whale,16,2.268508,2.195164,3.015644
1,bryant-stories.txt,ship,5,0.666072,0.606329,0.844081
2,bryant-stories.txt,captain,2,1.153544,0.953077,1.353293
3,whitman-leaves.txt,whale,5,1.896002,1.903065,2.285936
4,whitman-leaves.txt,ship,75,0.739784,0.740008,0.998119
5,whitman-leaves.txt,captain,19,1.432696,1.434334,1.877957


Con b=0, el libro de Bryant (46,321 tokens) pierde la ventaja por ser más corto que la media. Leaves of Grass (124,337 tokens) deja de recibir la penalización por longitud. Por eso intercambian los puestos 2 y 3.
Con k1=2, la saturación es más lenta. En el libro de Bryant, whale (TF=16) pasa de aportar 1.97 a 2.23 veces lo que aporta captain (TF=2). El cambio no invierte el orden de estos dos libros: Bryant sigue por encima con 5.213018 frente a 5.162012 de Whitman.


### Conclusiones

Comparé TF-IDF con similitud coseno y BM25 sobre los mismos 18 documentos de Gutenberg. Para `whale ship captain`, ambos métodos recuperaron los mismos diez documentos y ubicaron a *Moby Dick* en primer lugar. Sin embargo, el orden de los demás resultados cambió. Esto muestra que contar los documentos en común ayuda a comparar los rankings, pero también hace falta revisar sus posiciones.

La frecuencia documental (DF) me ayudó a interpretar cuántos resultados podía devolver cada consulta. `time` aparece en los 18 documentos, `queequeg` aparece en uno solo y ninguno de los términos de `thermonuclear microprocessor` está en el vocabulario. Por eso, la segunda consulta devolvió un documento y la última produjo dos listas vacías.

### Cómo influyó cada palabra en BM25

Para la búsqueda `whale ship captain`, *Persuasion* quedó segundo con TF-IDF. La palabra `captain` aparece 303 veces en ese libro, frente a solo 2 en *Stories to Tell to Children*.

Con BM25, el libro de Bryant subió del tercer al segundo puesto y *Persuasion* bajó al octavo. Esto pasó porque BM25 le da cada vez menos peso a las repeticiones de una misma palabra. Además, el libro de Bryant es más corto y contiene las tres palabras de la búsqueda. Solo `whale`, que aparece en 6 de los 18 libros, le dio 2,27 puntos, más que los 2,17 puntos que obtuvo *Persuasion* en total.

### Qué pasó al cambiar los parámetros

- **`b` ajusta la puntuación según la longitud del libro.** Al ponerlo en cero, el libro de Bryant perdió la ventaja que tenía por ser más corto. Por eso, *Leaves of Grass* pasó del tercer al segundo puesto.
- **`k₁` cambia cuánto pesan las repeticiones.** Al subirlo de 1,2 a 2, `whale`, que aparece 16 veces en el libro de Bryant, ganó más peso en comparación con `captain`, que aparece 2 veces. La puntuación cambió, pero el libro conservó el segundo puesto.